In [ ]:
# ==============================================================================
# PROGETTO: Forensic Network Analysis: The Epstein Corpus
# NOTEBOOK 3: Network Analytics & Web UI Export
# OBIETTIVI: 
# 1. Pruning e Data Cleaning post-NLP.
# 2. Costruzione del grafo orientato pesato (DiGraph) con NetworkX.
# 3. Calcolo metriche: Degree (Size), Betweenness Centrality (Brokers - RQ2).
# 4. Community Detection (Algoritmo di Louvain per i colori della UI).
# 5. Esportazione ottimizzata in JSON per React/WebGL.
# ==============================================================================

import pandas as pd
import networkx as nx
import community.community_louvain as community_louvain
import json
import numpy as np

# --- 1. CARICAMENTO E PULIZIA (SEMANTIC FILTERING) ---
print("1. Caricamento e filtraggio dei dati...")
df = pd.read_csv('../data/edges_classified.csv')

# Filtro 1: Rimuoviamo gli errori della pipeline NLP e le confidenze basse (< 65%)
# Perché: Vogliamo solo archi di cui siamo statisticamente certi.
df_clean = df[(df['semantic_category'] != 'ERROR') & (df['confidence'] >= 0.65)].copy()

# Filtro 2: Pulizia entità "spazzatura" (Hard Pruning)
# Rimuoviamo nodi anonimi o generici che falserebbero la topologia
junk_entities = ['unknown', 'unknown person a', 'unknown person b', 'staff member', 'himself', 'herself']
df_clean['actor_lower'] = df_clean['actor'].astype(str).str.lower()
df_clean['target_lower'] = df_clean['target'].astype(str).str.lower()

df_clean = df_clean[~df_clean['actor_lower'].isin(junk_entities)]
df_clean = df_clean[~df_clean['target_lower'].isin(junk_entities)]

In [ ]:
# --- 2. EDGE AGGREGATION (Ottimizzazione per il Web) ---
print("2. Aggregazione degli archi (Edge Weights)...")
# Se Epstein incontra Maxwell 100 volte, creiamo 1 solo arco di peso 100.
# Raggruppiamo per Attore, Target e Categoria Semantica
edges_aggregated = df_clean.groupby(['actor', 'target', 'semantic_category']).size().reset_index(name='weight')

In [ ]:
# --- 3. COSTRUZIONE DEL GRAFO (NetworkX) ---
print("3. Costruzione della topologia di rete...")
G = nx.DiGraph()

# Aggiungiamo gli archi (i nodi vengono creati automaticamente)
for _, row in edges_aggregated.iterrows():
    G.add_edge(row['actor'], row['target'], weight=row['weight'], category=row['semantic_category'])

print(f"Grafo grezzo creato: {G.number_of_nodes()} Nodi, {G.number_of_edges()} Archi.")

In [ ]:
# --- 4. CALCOLO DELLE METRICHE CENTRALI (La Matematica della Tesi) ---
print("4. Calcolo delle metriche di Centrality e Community Detection...")

# A. Degree Centrality (Quanto è grande il nodo visivamente?)
# Calcoliamo il grado totale (in-degree + out-degree)
degree_dict = dict(G.degree(weight='weight'))

# B. Betweenness Centrality (La risposta alla RQ2: I Brokers)
# Chi fa da ponte tra mondi isolati? (Calcolo intensivo, usiamo il grafo non pesato per la topologia)
betweenness_dict = nx.betweenness_centrality(G, normalized=True)

# C. Community Detection (Louvain Modularity)
# L'algoritmo di Louvain lavora su grafi non orientati.
G_undirected = G.to_undirected()
partition = community_louvain.best_partition(G_undirected, weight='weight')

In [ ]:
# --- 5. PRUNING PER IL BROWSER E FORMATTAZIONE JSON ---
print("5. Pruning e generazione del formato JSON per la Web UI...")

# Per evitare che il browser esploda, teniamo solo i nodi con almeno X interazioni totali
# (Regola questo parametro MIN_DEGREE se il grafo è ancora troppo denso)
MIN_DEGREE = 3  
valid_nodes = {node for node, deg in degree_dict.items() if deg >= MIN_DEGREE}

nodes_data = []
for node in valid_nodes:
    nodes_data.append({
        "id": node,
        "label": node,
        "val": degree_dict[node],                    # Usato per la dimensione del nodo (Size)
        "betweenness": betweenness_dict[node],       # Metrica per l'analisi visiva avanzata
        "group": partition[node]                     # Usato per colorare il nodo (Cluster)
    })

links_data = []
for u, v, data in G.edges(data=True):
    # Aggiungiamo l'arco solo se ENTRAMBI i nodi sono sopravvissuti al taglio (MIN_DEGREE)
    if u in valid_nodes and v in valid_nodes:
        links_data.append({
            "source": u,
            "target": v,
            "weight": data['weight'],                # Spessore della linea
            "category": data['category']             # Colore della linea (Logistico, Sociale, Legale)
        })

web_graph = {
    "nodes": nodes_data,
    "links": links_data
}

# Salvataggio su disco
output_json_path = '../data/web_graph_data.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(web_graph, f, ensure_ascii=False, indent=2)

print(f"\n✅ Pipeline completata con successo!")
print(f"📊 Nodi esportati per la UI: {len(nodes_data)}")
print(f"🔗 Archi esportati per la UI: {len(links_data)}")
print(f"📁 File pronto per React/WebGL salvato in: {output_json_path}")